In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/augdatasetusingst1/st1_aug_full_label1.csv
/kaggle/input/augdatasetusingst1/augmented_data_label0.csv
/kaggle/input/augdatasetusingst1/st1.csv
/kaggle/input/augdatasetusingst1/augmented_data_label1.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/tel.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/mya.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/amh.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/eng.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/ben.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/hau.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/khm.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/pan.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/pol.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/arb.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/hin.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/nep.csv
/kaggle/input/update-ds-semeval2026/subtask2/test/ita.csv
/kaggle/input/update-ds-semeva

In [2]:
!pip install -U bitsandbytes accelerate 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 34.1 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [3]:
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score,recall_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    set_seed
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import EarlyStoppingCallback

2026-02-06 08:55:51.298477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770368151.510394      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770368151.574302      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770368152.078909      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770368152.078951      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770368152.078954      24 computation_placer.cc:177] computation placer alr

In [ ]:
# hf_token = "your_hf_token_here"
print("HF_TOKEN exists:", hf_token is not None)


HF_TOKEN exists: True


In [5]:
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"

SEED = 42
MAX_LEN = 256

TEXT_COL = "text"
LABEL_COL = "polarization"

OUTPUT_DIR = "/kaggle/working/qlora_llama31_8b_polar"

set_seed(SEED)

In [6]:
df_original = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/train/eng.csv")
df_1 = pd.read_csv("/kaggle/input/augdatasetusingst1/augmented_data_label0.csv")
df_2 = pd.read_csv("/kaggle/input/augdatasetusingst1/augmented_data_label1.csv")
df_3 = pd.read_csv("/kaggle/input/augdatasetusingst1/st1_aug_full_label1.csv")


In [7]:
print(df_2.shape)

(1096, 2)


In [8]:
df_1_sample = df_1.sample(n=850, random_state=42)
df_2_sample = df_2.sample(n=850, random_state=42)
# df_3_sample = df_3.sample(n=850, random_state=42)
df = pd.concat([df_1_sample, df_2_sample, df_original], ignore_index=True)
# df = df_original
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df[LABEL_COL]
)

train_ds = Dataset.from_pandas(df_train.reset_index(drop=True))
val_ds   = Dataset.from_pandas(df_val.reset_index(drop=True))


In [9]:
df["polarization"].value_counts()

polarization
0    2897
1    2025
Name: count, dtype: int64

In [10]:
df_train.shape


(3937, 3)

In [11]:
df_val.shape

(985, 3)

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16  # Kaggle A100/L4/T4: bfloat16 thường ok; nếu lỗi đổi float16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    token=hf_token
)

# Llama Instruct thường không có pad_token -> set = eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    token=hf_token
)

model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

# LoRA config 
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 13,639,680 || all params: 7,518,572,544 || trainable%: 0.1814


In [14]:
def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
    )

train_tok = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[TEXT_COL, "id"]
)

val_tok = val_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[TEXT_COL, "id"]
)

train_tok = train_tok.rename_column(LABEL_COL, "labels")
val_tok   = val_tok.rename_column(LABEL_COL, "labels")


Map:   0%|          | 0/3937 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="binary", zero_division=0),
        "recall": recall_score(labels, preds, average="binary", zero_division=0),
    }


In [17]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,

    # batch
    per_device_train_batch_size=4,      # nếu GPU mạnh (A100) có thể tăng 4
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,      # effective batch ~16
    num_train_epochs=2,

    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,

    # eval/save
    eval_strategy="steps",
    eval_steps=80,
    save_strategy="steps",
    save_steps=80,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # logging
    logging_steps=20,
    report_to="none",

    # perf
    bf16=True,                          # nếu lỗi -> đổi bf16=False và fp16=True
    fp16=False,
    optim="paged_adamw_8bit",
)


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=4,      # số lần eval không cải thiện
        early_stopping_threshold=0.0    # cải thiện tối thiểu
    )]
)


trainer.train()


/tmp/ipykernel_24/2548175638.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision,Recall
80,1.776500,0.373793,0.857868,0.850043,0.856129,0.873239,0.765432
160,1.660800,0.348075,0.849746,0.839489,0.846698,0.888218,0.725926
240,1.391900,0.280864,0.871066,0.866083,0.870673,0.856410,0.824691
320,0.867100,0.366105,0.875127,0.869531,0.874331,0.875000,0.812346
400,0.479200,0.403147,0.876142,0.869824,0.874919,0.889807,0.797531
480,0.721100,0.389727,0.881218,0.877115,0.881104,0.860000,0.849383


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6985b1df-0d47c6404765ba8f15d49254;846c2d5f-6ec2-4979-8a9b-3cbcb589bdb7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:286: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will 

TrainOutput(global_step=494, training_loss=1.29679048013108, metrics={'train_runtime': 7191.4965, 'train_samples_per_second': 1.095, 'train_steps_per_second': 0.069, 'total_flos': 8305530683867136.0, 'train_loss': 1.29679048013108, 'epoch': 2.0})

In [19]:
metrics = trainer.evaluate()
print(metrics)

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)


{'eval_loss': 0.38972681760787964, 'eval_accuracy': 0.8812182741116751, 'eval_f1_macro': 0.8771146002719057, 'eval_f1_weighted': 0.8811042831716815, 'eval_precision': 0.86, 'eval_recall': 0.8493827160493828, 'eval_runtime': 238.3219, 'eval_samples_per_second': 4.133, 'eval_steps_per_second': 0.52, 'epoch': 2.0}


/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1228: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6985ca53-250b1ded310d8279635281ae;00d0b7ef-7614-4732-9f53-944242cd73f7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3.1-8B.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:286: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3.1-8B - will assume that the vocabulary was not modified.
  warnings.warn(


Saved to: /kaggle/working/qlora_llama31_8b_polar


In [20]:
def predict_one(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        out = model(**inputs)
        probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
    pred = int(np.argmax(probs))
    return pred, probs

samples = [
    "Democrats are so disgusting. Do better!",
    "I love pizza and going to the beach.",
    "The woke mob is trying to erase our history."
]

for s in samples:
    pred, probs = predict_one(s)
    print("\nTEXT:", s)
    print("PRED:", pred, "PROBS:", probs)



TEXT: Democrats are so disgusting. Do better!
PRED: 1 PROBS: [4.4001186e-05 9.9995601e-01]

TEXT: I love pizza and going to the beach.
PRED: 0 PROBS: [9.9908900e-01 9.1105123e-04]

TEXT: The woke mob is trying to erase our history.
PRED: 1 PROBS: [1.0129990e-05 9.9998987e-01]


In [21]:
import numpy as np
import torch
import pandas as pd
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import os

# 1. Load Data
df_test = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/test/eng.csv")
df_dev = pd.read_csv("/kaggle/input/update-ds-semeval2026/subtask1/dev/eng.csv")

# 2. Preparation
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
    )

# --- PROCESS DEV SET ---
print("Processing Dev Set...")
dev_ds = Dataset.from_pandas(df_dev.reset_index(drop=True))

# Tokenize and format columns for Dev
dev_tok = dev_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text", "id"]
)

# Rename label column if it exists to 'labels' for evaluation
if "polarization" in dev_tok.column_names:
    dev_tok = dev_tok.rename_column("polarization", "labels")

# Predict on Dev
dev_output = trainer.predict(dev_tok)
dev_logits = dev_output.predictions
dev_preds = np.argmax(dev_logits, axis=-1)

# Calculate Metrics for Dev
if "labels" in dev_tok.column_names:
    dev_labels = dev_output.label_ids
    
    acc = accuracy_score(dev_labels, dev_preds)
    f1 = f1_score(dev_labels, dev_preds, average="macro")
    prec = precision_score(dev_labels, dev_preds, average="binary", zero_division=0)
    rec = recall_score(dev_labels, dev_preds, average="binary", zero_division=0)
    
    print("\n=== Dev Set Metrics ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Macro:  {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print("=======================\n")

# Save Dev Predictions
submission_dev = pd.DataFrame({
    "id": df_dev["id"],
    "polarization": dev_preds,
})
dev_output_path = os.path.join(OUTPUT_DIR, "pred_eng_dev.csv")
submission_dev.to_csv(dev_output_path, index=False)
print(f"Dev predictions saved to: {dev_output_path}")


# --- PROCESS TEST SET ---
print("Processing Test Set...")
test_ds = Dataset.from_pandas(df_test.reset_index(drop=True))

test_tok = test_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text", "id"]
)

# Predict on Test
pred_output = trainer.predict(test_tok)
logits = pred_output.predictions

# Calculate probabilities
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
test_preds = np.argmax(probs, axis=1)

# Save Test Predictions
submission = pd.DataFrame({
    "id": df_test["id"],
    "polarization": test_preds,
})

test_output_path = os.path.join(OUTPUT_DIR, "pred_eng.csv")
submission.to_csv(test_output_path, index=False)

# Save Probabilities
submission_probs = pd.DataFrame({
    "id": df_test["id"],
    "prob_class1": probs[:, 1],
})
probs_output_path = os.path.join(OUTPUT_DIR, "pred_eng_probs.csv")
submission_probs.to_csv(probs_output_path, index=False)

print(f"Test predictions saved to: {test_output_path}")
print(f"Test probabilities saved to: {probs_output_path}")

Processing Dev Set...


Map:   0%|          | 0/160 [00:00<?, ? examples/s]


=== Dev Set Metrics ===
Accuracy:  0.8250
F1 Macro:  0.8107
Precision: 0.7719
Recall:    0.7458

Dev predictions saved to: /kaggle/working/qlora_llama31_8b_polar/pred_eng_dev.csv
Processing Test Set...


Map:   0%|          | 0/1452 [00:00<?, ? examples/s]

Test predictions saved to: /kaggle/working/qlora_llama31_8b_polar/pred_eng.csv
Test probabilities saved to: /kaggle/working/qlora_llama31_8b_polar/pred_eng_probs.csv
